# Week 3 exercises: Loading Data, Databases, APIs, and Joining & Reshaping

*Complete all exercises in this notebook during week 3. The exercises are not submitted — this is purely for your own learning and for preparing for the assignments, project and exam.*

Note! You should always complete the tasks first without using AI. If you need AI to get through these tasks, then you will not be able to pass the oral exam. AI can be used to check if your code could be simplified or improved after you solve the task.

---

## Part 1: Storing API Keys Securely

### Exercise 1: Creating and using a .env file

When working with APIs that require authentication, you need an API key. API keys must **never** be written directly in your code or notebook, because:
- If you upload your notebook to GitHub, anyone can see and [misuse your key](https://www.theregister.com/2026/03/03/gemini_api_key_82314_dollar_charge/).
- Sharing your notebook with a classmate would also expose the key (what if they accidentally share it on Github?).

A simple and safe approach is to store your key in a separate `.env` (env for environment) file that is **excluded from version control**.

a) Create a file called `.env` in the same folder as this notebook (you can do this with a text editor, VS Code, or from Python). I recommend [Sublime Text](https://www.sublimetext.com/) as a convenient, light and free way to view and create practically any file needed in software development. The file should have the following content:

```
# .env (local only, never commit)
GEMINI_API_KEY=YOUR_API_KEY_HERE
```

Replace `YOUR_API_KEY_HERE` with your actual API key. For this exercise, we will use Google's Gemini API as an example (you can obtain one from [Google AI Studio](https://aistudio.google.com/apikey)).

b) Add `.env` to your `.gitignore` file so that the key is never uploaded to GitHub. You can do this by opening (or creating) `.gitignore` in a text editor and adding a line containing `.env`. **Do not skip this step.**

## Exercise 2: Reading the API key from the .env file

The `python-dotenv` package automates reading variables that store information such as API keys from the .env file and is the standard approach in Python projects. It loads all variables from a `.env` file into the environment in a single call.

Run the code below to make sure you have stored the API key correctly.

**Note:** You may need to install the package first with `pip install python-dotenv`. 

In [ ]:
# !pip install python-dotenv

Defaulting to user installation because normal site-packages is not writeable


In [42]:
from dotenv import load_dotenv
import os

# Load all variables from .env into the environment
load_dotenv()

api_key = os.environ.get("DEMINI_API_KEY")

if api_key:
    print(f"Key loaded: {api_key[:3]}...")
else:
    print("ERROR: GEMINI_API_KEY not found. Check your .env file.")

Key loaded: AIz...


## Part 2: Loading and Exporting Data

### Exercise 3: Loading CSV data

The file `data/cafe_sales.csv` contains sales data for a small café.

a) Load the file into a pandas DataFrame. Print the shape and the first 5 rows.

b) Load only the columns `product`, `category`, and `revenue` using the `usecols` parameter. Set `product` as the index. Print the resulting DataFrame.

c) Export the DataFrame from (b) to a new CSV file called `data/category_revenue.csv` without the index column. Then reload the exported file and print it to verify the export was correct.

In [27]:
# YOUR CODE HERE

# a)

import pandas as pd
df = pd.read_csv("data/cafe_sales.csv")
print("Shape", df.shape)
print()
print(df.head())

Shape (12, 5)

      product category  units_sold  unit_price  revenue
0    Espresso   Coffee         320         3.5   1120.0
1  Cappuccino   Coffee         280         4.5   1260.0
2       Latte   Coffee         250         4.9   1225.0
3   Green Tea      Tea         180         3.2    576.0
4  Chai Latte      Tea         150         4.7    705.0


In [44]:
# b)

df_subset = pd.read_csv(
    "data/cafe_sales.csv",
    usecols=["product", "category", "revenue"],
    index_col="product",
    )

df_subset

,category,revenue
product,,
Espresso,Coffee,1120.0
Cappuccino,Coffee,1260.0
Latte,Coffee,1225.0
Green Tea,Tea,576.0
Chai Latte,Tea,705.0
Croissant,Pastry,760.0
Blueberry Muffin,Pastry,714.0
Cinnamon Roll,Pastry,585.0
Iced Americano,Coffee,840.0


In [45]:
# c)

df_subset.to_csv(
    "data/category_revenue.csv",
    index=False
)

df_subset_loaded = pd.read_csv(
    "data/category_revenue.csv"
)

df_subset_loaded

,category,revenue
0,Coffee,1120.0
1,Coffee,1260.0
2,Coffee,1225.0
3,Tea,576.0
4,Tea,705.0
5,Pastry,760.0
6,Pastry,714.0
7,Pastry,585.0
8,Coffee,840.0
9,Tea,624.0


### Exercise 4: Working with JSON data

The file `data/projects.json` contains project records for a small company. Each project has a list of tasks with estimated hours and a status.

a) Load the file using the `json` module. Print the number of projects and the full first record.

b) Use `pd.json_normalize()` to flatten the nested tasks into a regular DataFrame. The result should have one row per task and include the `project_id` and `title` from the parent record. Print the DataFrame. The documentation for pd.json_normalice() is found [here](https://pandas.pydata.org/docs/reference/api/pandas.json_normalize.html).

c) Export the flattened tasks DataFrame to a JSON file called `data/tasks_flat.json` using the `records` orientation. Load it again and print it to see that it was saved correctly.

In [ ]:
# YOUR CODE HERE

# a)

import json

with open("data/projects.json", "r") as f:
    projects = json.load(f)

print("Number of projects:", len(projects))
print()
print("First record:")
print(projects[0])

Number of projects: 3

First record:
{'project_id': 'P001', 'title': 'Website Redesign', 'manager': 'Laura', 'tasks': [{'task': 'Wireframing', 'hours': 20, 'status': 'completed'}, {'task': 'Frontend development', 'hours': 60, 'status': 'in_progress'}, {'task': 'User testing', 'hours': 15, 'status': 'not_started'}]}


In [ ]:
# b)

projects_normalized = pd.json_normalize(
    projects,
    record_path="tasks",
    meta=["project_id", "title"])

projects_normalized

,task,hours,status,project_id,title
0,Wireframing,20,completed,P001,Website Redesign
1,Frontend development,60,in_progress,P001,Website Redesign
2,User testing,15,not_started,P001,Website Redesign
3,UI design,35,completed,P002,Mobile App Launch
4,Backend API,80,in_progress,P002,Mobile App Launch
5,App store submission,5,not_started,P002,Mobile App Launch
6,Beta testing,25,not_started,P002,Mobile App Launch
7,Schema mapping,15,completed,P003,Data Pipeline Migration
8,ETL scripts,45,completed,P003,Data Pipeline Migration
9,Validation,20,in_progress,P003,Data Pipeline Migration


In [83]:
# c)

projects_normalized.to_json("data/tasks_flat.json", orient="records", indent=2)

pd.read_json("data/tasks_flat.json")

,task,hours,status,project_id,title
0,Wireframing,20,completed,P001,Website Redesign
1,Frontend development,60,in_progress,P001,Website Redesign
2,User testing,15,not_started,P001,Website Redesign
3,UI design,35,completed,P002,Mobile App Launch
4,Backend API,80,in_progress,P002,Mobile App Launch
5,App store submission,5,not_started,P002,Mobile App Launch
6,Beta testing,25,not_started,P002,Mobile App Launch
7,Schema mapping,15,completed,P003,Data Pipeline Migration
8,ETL scripts,45,completed,P003,Data Pipeline Migration
9,Validation,20,in_progress,P003,Data Pipeline Migration


## Part 3: Databases

### Exercise 5: Connecting and querying a SQLite database

The file `data/retail.db` is a SQLite database with tables including Product, Vendor, Store, Region, Customer, SalesTransaction, and Includes. The tables, relationships and a sample of the data are visualized below:

<img src="data/schema.png" alt="Database schema" width="700">


a) Connect to the database using `sqlite3`. List all table names by querying `sqlite_master`.

b) Using `fetchall()`, retrieve and print the names and prices of all products that cost more than 100€. Format the output so that each product is printed as `ProductName: Price€`.

c) Using a parameterised query (with `?` placeholders), retrieve all products supplied by vendor `"MK"`. Print the product names.

In [88]:
# YOUR CODE HERE

import sqlite3

# a)

connection = sqlite3.connect("data/retail.db")
db = connection.cursor()

db.execute("SELECT name FROM sqlite_schema WHERE type='table';")
tables = db.fetchall()

for table in tables:
    print(table[0])

connection.close()

vendor
category
product
region
store
customer
salestransaction
includes


In [94]:
# b)
connection = sqlite3.connect("data/retail.db")
db = connection.cursor()

db.execute("SELECT ProductName, ProductPrice FROM PRODUCT WHERE ProductPrice > 100")
rows = db.fetchall()

for name, price in rows:
    print(f"{name}: {price}€")

connection.close()

Tiny Tent: 150€
Biggy Tent: 250€
Hi-Tec GPS: 300€
Comfy Harness: 150€
Sunny Charger: 125€
Mega Camera: 275€
Luxo Tent: 500€


In [105]:
# c)
connection = sqlite3.connect("data/retail.db")
db = connection.cursor()

vendor = "MK"

db.execute("SELECT ProductName FROM PRODUCT WHERE VendorID=?", [vendor])
products_by_vendor = db.fetchall()

connection.close()

for product in products_by_vendor:
    print(product[0])

Easy Boot
Cosy Sock
Tiny Tent
Biggy Tent
Power Pedals
Comfy Harness
Strongster Carribeaner
Electra Compass


### Exercise 6: Loading database results into pandas

Using the same `retail.db` database:

a) Use `pd.read_sql_query()` to load the entire `Customer` table into a DataFrame. Print the shape and the first 5 rows.

b) Write a SQL query that joins the `SalesTransaction` and `Customer` tables on `CustomerID` and returns `CustomerName`, `TDate`, and `StoreID`. Load the result into a DataFrame and print the first 5 rows. If you have not taken an introductory database course, use an AI chatbot to solve this part.

c) Close the database connection.

In [8]:
# YOUR CODE HERE


## Part 4: APIs

### Exercise 7: Fetching data from a public API

a) Use the `requests` library to fetch the latest EUR exchange rates from:  
`https://open.er-api.com/v6/latest/EUR`

Check that the status code is 200 and print the exchange rates for USD, GBP, and SEK.

b) Convert the full rates dictionary into a pandas DataFrame with columns `currency` and `rate`. Print the first 10 rows.

c) Using the Open-Meteo API, fetch the 7-day forecast for Turku (latitude 60.45, longitude 22.27) with `temperature_2m_max` and `precipitation_sum` as daily variables and `Europe/Helsinki` as the timezone. Convert the result into a DataFrame with columns `date`, `temp_max`, and `precipitation`. Print the DataFrame.

In [73]:
# YOUR CODE HERE

import requests

# a)

url = "https://open.er-api.com/v6/latest/EUR"
response = requests.get(url)
print("Status code:", response)

data = response.json()
print("USD:", data["rates"]["USD"])
print("GBP:", data["rates"]["GBP"])
print("SEK:", data["rates"]["SEK"])

Status code: <Response [200]>
USD: 1.169015
GBP: 0.870676
SEK: 10.861204


In [ ]:
# b)

import pandas as pd

df = pd.DataFrame({
    "currency" = data["rates"],
    
})

df

,result,provider,documentation,terms_of_use,time_last_update_unix,time_last_update_utc,time_next_update_unix,time_next_update_utc,time_eol_unix,base_code,rates
EUR,success,https://www.exchangerate-api.com,https://www.exchangerate-api.com/docs/free,https://www.exchangerate-api.com/terms,1775779352,"Fri, 10 Apr 2026 00:02:32 +0000",1775866542,"Sat, 11 Apr 2026 00:15:42 +0000",0,EUR,1.000000
AED,success,https://www.exchangerate-api.com,https://www.exchangerate-api.com/docs/free,https://www.exchangerate-api.com/terms,1775779352,"Fri, 10 Apr 2026 00:02:32 +0000",1775866542,"Sat, 11 Apr 2026 00:15:42 +0000",0,EUR,4.293193
AFN,success,https://www.exchangerate-api.com,https://www.exchangerate-api.com/docs/free,https://www.exchangerate-api.com/terms,1775779352,"Fri, 10 Apr 2026 00:02:32 +0000",1775866542,"Sat, 11 Apr 2026 00:15:42 +0000",0,EUR,74.938994
ALL,success,https://www.exchangerate-api.com,https://www.exchangerate-api.com/docs/free,https://www.exchangerate-api.com/terms,1775779352,"Fri, 10 Apr 2026 00:02:32 +0000",1775866542,"Sat, 11 Apr 2026 00:15:42 +0000",0,EUR,95.855398
AMD,success,https://www.exchangerate-api.com,https://www.exchangerate-api.com/docs/free,https://www.exchangerate-api.com/terms,1775779352,"Fri, 10 Apr 2026 00:02:32 +0000",1775866542,"Sat, 11 Apr 2026 00:15:42 +0000",0,EUR,439.099465
...,...,...,...,...,...,...,...,...,...,...,...
YER,success,https://www.exchangerate-api.com,https://www.exchangerate-api.com/docs/free,https://www.exchangerate-api.com/terms,1775779352,"Fri, 10 Apr 2026 00:02:32 +0000",1775866542,"Sat, 11 Apr 2026 00:15:42 +0000",0,EUR,278.460341
ZAR,success,https://www.exchangerate-api.com,https://www.exchangerate-api.com/docs/free,https://www.exchangerate-api.com/terms,1775779352,"Fri, 10 Apr 2026 00:02:32 +0000",1775866542,"Sat, 11 Apr 2026 00:15:42 +0000",0,EUR,19.165133
ZMW,success,https://www.exchangerate-api.com,https://www.exchangerate-api.com/docs/free,https://www.exchangerate-api.com/terms,1775779352,"Fri, 10 Apr 2026 00:02:32 +0000",1775866542,"Sat, 11 Apr 2026 00:15:42 +0000",0,EUR,22.458725
ZWG,success,https://www.exchangerate-api.com,https://www.exchangerate-api.com/docs/free,https://www.exchangerate-api.com/terms,1775779352,"Fri, 10 Apr 2026 00:02:32 +0000",1775866542,"Sat, 11 Apr 2026 00:15:42 +0000",0,EUR,29.437600


### Exercise 8: Using an API key with Gemini

This exercise uses the Gemini API key that you stored in `.env` in Exercise 1.

a) Load your API key from `.env` using either approach from Part 1.

b) Install the `google-genai` package (if needed) and use it to send a simple prompt to Gemini. For example, ask it to explain what an API is in one sentence. Print the response.

```python
from google import genai

client = genai.Client(api_key=api_key)
response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents="Explain what an API is in one sentence."
)
print(response.text)
```

c) Write a function called `ask_gemini(prompt)` that takes a prompt string as input, sends it to the Gemini API, and returns the response text. Test it with a prompt of your choice.

In [ ]:
# YOUR CODE HERE

# a)

## Part 5: Joining and Reshaping Data

### Exercise 9: Merging DataFrames

Consider the following two DataFrames:

```python
stores = pd.DataFrame({
    "store_id": [1, 2, 3, 4],
    "city": ["Helsinki", "Turku", "Tampere", "Oulu"],
    "region": ["South", "South", "Central", "North"]
})

sales = pd.DataFrame({
    "transaction_id": [1001, 1002, 1003, 1004, 1005],
    "store_id": [1, 2, 2, 3, 5],
    "amount": [250, 120, 310, 180, 90]
})
```

a) Perform an **inner join** on `store_id`. How many rows are in the result and why?

b) Perform a **left join** with `stores` on the left. Which store has no sales and how is it represented?

c) Perform an **outer join**. Identify which rows have `NaN` values and explain why.

In [ ]:
# YOUR CODE HERE


### Exercise 10: Concatenation and reshaping

a) Create two DataFrames representing monthly sales for two quarters:

```python
q1 = pd.DataFrame({
    "month": ["Jan", "Feb", "Mar"],
    "store_A": [20000, 22000, 21000],
    "store_B": [15000, 16000, 17000]
})

q2 = pd.DataFrame({
    "month": ["Apr", "May", "Jun"],
    "store_A": [23000, 24000, 25000],
    "store_B": [17500, 18000, 19000]
})
```

Concatenate them vertically into a single DataFrame with `ignore_index=True`. Print the result.

b) Use `melt()` to convert the concatenated DataFrame from wide to long format. The `month` column should remain as the identifier, and the store columns should become rows. Use `store` as the variable name and `sales` as the value name. Print the result.

c) Use `pivot()` to convert the long DataFrame back to wide format. Verify that the result matches the original concatenated DataFrame.

In [ ]:
# YOUR CODE HERE
